# How many failure types are still undiscovered?

This notebook runs the pilot on Colab using every CPU core in parallel. No GPU is needed: choose **Runtime → Change runtime type → CPU** (High-RAM if available).

Run the cells in order. The pilot (3 datasets × 3 models × 5 seeds) takes about 10–20 minutes on 8 cores.

In [1]:
# 1. Get the code and check it
!git clone -q https://github.com/ParitKansal/undiscovered-failures.git
%cd undiscovered-failures
!pip install -q -r requirements.txt
!python -m tests.test_estimators
!python -m tests.test_end_to_end | tail -1
import os; print('CPU cores:', os.cpu_count())

/content/undiscovered-failures
ALL 12 ESTIMATOR CHECKS PASSED
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   8 out of   8 | elapsed:    2.7s finished
END-TO-END TEST PASSED
CPU cores: 24


In [2]:
# 2. Optional: save results to Google Drive so they survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/undiscovered-failures'
os.makedirs(OUT, exist_ok=True)

Mounted at /content/drive


In [3]:
# 3. Run the pilot in parallel (all cores)
!python experiment.py --datasets adult,bank,covertype --models logreg,forest,boosting --seeds 5 --repeats 30 --out {OUT}/pilot.csv --data_home /content/data

45 jobs on 24 CPU cores
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 out of  45 | elapsed:   26.7s remaining:  2.1min
[Parallel(n_jobs=-1)]: Done  18 out of  45 | elapsed:   38.1s remaining:   57.1s
[Parallel(n_jobs=-1)]: Done  28 out of  45 | elapsed:   48.3s remaining:   29.3s
[Parallel(n_jobs=-1)]: Done  38 out of  45 | elapsed:   58.3s remaining:   10.7s
[Parallel(n_jobs=-1)]: Done  45 out of  45 | elapsed:  1.2min finished
wrote /content/drive/MyDrive/undiscovered-failures/pilot.csv 16200 rows


In [4]:
# 4. Summary tables
!python summarize.py {OUT}/pilot.csv

== By type definition and test budget ==
                 rel_s_obs  rel_chao1  rel_ichao1  rel_ace  rel_jackknife1  cov_chao1_ln  cov_chao1_b  cov_ace_b  cov_jack_b  new_real  new_ratio  true_types
types    budget                                                                                                                                              
kmeans10 300        -0.254     -0.219      -0.209   -0.164          -0.154         0.306        0.421      0.552       0.576     0.589      0.446      10.000
         1000       -0.157     -0.137      -0.134   -0.090          -0.092         0.401        0.495      0.632       0.659     0.441      0.332      10.000
         3000       -0.091     -0.065      -0.062   -0.011          -0.028         0.513        0.613      0.724       0.751     0.408      0.437      10.000
kmeans25 300        -0.341     -0.227      -0.203   -0.208          -0.175         0.812        0.507      0.492       0.512     2.409      0.780      25.000
         10

## Pilot v1: wider design and bootstrap intervals
4 datasets (adds MAGIC) × 3 models × 3 seeds × 2 flip rates (0.6, 0.9) × 3 numbers of planted types (15, 30, 60) = 216 jobs.
Each row now also has bootstrap 95% intervals for Chao1, ACE and the jackknife. About 30–40 minutes on 8 cores (faster with more).

In [5]:
# 5. Run pilot v1 (all cores)
!git pull -q
!python -m tests.test_estimators
!python experiment.py --datasets adult,bank,covertype,magic --models logreg,forest,boosting --seeds 3 --repeats 30 --flips 0.6,0.9 --k_planted 15,30,60 --boot 200 --out {OUT}/pilot_v1.csv --data_home /content/data

ALL 12 ESTIMATOR CHECKS PASSED
216 jobs on 24 CPU cores
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:   53.8s
[Parallel(n_jobs=-1)]: Done 114 tasks      | elapsed:  2.5min
/usr/local/lib/python3.13/dist-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
[Parallel(n_jobs=-1)]: Done 213 out of 216 | elapsed:  3.6min remaining:    3.1s
[Parallel(n_jobs=-1)]: Done 216 out of 216 | elapsed:  3.7min finished
wrote /content/drive/MyDrive/undiscovered-failures/pilot_v1.csv 77760 rows


In [6]:
# 6. Summary tables for v1
!python summarize.py {OUT}/pilot_v1.csv

== By type definition and test budget ==
                 rel_s_obs  rel_chao1  rel_ichao1  rel_ace  rel_jackknife1  cov_chao1_ln  cov_chao1_b  cov_ace_b  cov_jack_b  new_real  new_ratio  true_types
types    budget                                                                                                                                              
kmeans10 300        -0.197     -0.166      -0.155   -0.121          -0.101         0.380        0.538      0.658       0.691     0.550      0.431      10.000
         1000       -0.113     -0.102      -0.100   -0.068          -0.065         0.517        0.591      0.692       0.706     0.338      0.261      10.000
         3000       -0.062     -0.047      -0.046   -0.011          -0.018         0.658        0.746      0.817       0.832     0.295      0.378      10.000
kmeans25 300        -0.317     -0.197      -0.171   -0.173          -0.138         0.828        0.588      0.588       0.616     2.644      0.764      25.000
         10